# Video format

> Like the folder format, with image columns as one MP4 per episode (and per agent) (stable-worldmodel's `video` format).

In [ ]:
#| default_exp data.video

In [ ]:
#| hide
from nbdev.showdoc import *

Layout: as the folder format, with image columns as `<img_col>/ep_<i>.mp4` (one video per
episode) or, per agent, `<img_col>/ep_<i>_agent_<a>.mp4`. Frames are decoded with torchcodec
when it is installed (as stable-worldmodel), else with imageio. Video is lossy. Sizes are kept
when they are even (stable-worldmodel's writer rounds them up to multiples of 16, which changes
the 56 or 28 pixel views of gridworlds), odd ones are rounded to the next even size.

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from functools import lru_cache
from pathlib import Path
from typing import Any

import numpy as np

from stable_marl.data.dataset import to_tensor
from stable_marl.data.folder import FolderDataset, _NpzEpisodeWriter
from stable_marl.data.format import Format, register_format

In [ ]:
#| export
def _decode(path: str) -> np.ndarray:
    "Every frame of a video file, `(T, H, W, C)` uint8 RGB (torchcodec if installed, else imageio)."
    try:
        from torchcodec.decoders import VideoDecoder
        return VideoDecoder(path, seek_mode='approximate')[:].permute(0, 2, 3, 1).numpy()
    except ImportError:
        import imageio.v3 as iio
        return np.stack(list(iio.imiter(path)))


class VideoDataset(FolderDataset):
    "Dataset of a video folder: image columns decoded from one MP4 per episode (and per agent)."
    def __init__(self, name: str | None = None, video_keys: list[str] | None = None, **kw: Any):
        super().__init__(name=name, folder_keys=video_keys, **kw)
        self._agents = {k: self._video_agents(k) for k in self.folder_keys if (self.path / k).is_dir()}

    def _video_agents(self, key: str) -> int | None:
        names = [p.name for p in (self.path / key).glob('ep_*_agent_*.mp4')]
        return max(int(n.rsplit('_agent_', 1)[1][:-4]) for n in names) + 1 if names else None

    @lru_cache(maxsize=8)
    def _frames(self, ep_idx: int, key: str, agent: int | None) -> np.ndarray:
        name = f'ep_{ep_idx}.mp4' if agent is None else f'ep_{ep_idx}_agent_{agent}.mp4'
        return _decode(str(self.path / key / name))

    def _load_file(self, ep_idx: int, step: int, key: str) -> np.ndarray:
        A = self._agents.get(key)
        if A is None:
            return self._frames(ep_idx, key, None)[step]
        return np.stack([self._frames(ep_idx, key, a)[step] for a in range(A)])

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        g_start, g_end = self.offsets[ep_idx] + start, self.offsets[ep_idx] + end
        steps, idx = {}, list(range(start, end, self.frameskip))
        for col in self._keys:
            if col in self.folder_keys:
                A = self._agents.get(col)
                frames = self._frames(ep_idx, col, None)[idx] if A is None else \
                    np.stack([self._frames(ep_idx, col, a)[idx] for a in range(A)], axis=1)
                steps[col] = to_tensor(frames)               # (T, C, H, W), or per agent (T, A, H, W, C)
            else:
                data = self._cache[col][g_start:g_end]
                if col != 'action':
                    data = data[::self.frameskip]
                if data.dtype == np.object_ or data.dtype.kind in ('S', 'U'):
                    val = data[0] if len(data) > 0 else b''
                    steps[col] = val.decode() if isinstance(val, bytes) else val
                else:
                    steps[col] = to_tensor(data)
        return self.transform(steps) if self.transform else steps


class VideoWriter(_NpzEpisodeWriter):
    """
    Appends episodes; image columns are encoded as one MP4 per episode (and per agent), with
    `fps` and `codec`. `mode`: `'append'` (default), `'overwrite'` or `'error'`.
    """
    label = 'VideoWriter'

    def __init__(self, path, fps: int = 25, codec: str = 'libx264', *, mode: str = 'append'):
        super().__init__(path, mode=mode)
        self.fps, self.codec = fps, codec

    def _is_image_dir(self, sub: Path) -> bool:
        return sub.is_dir() and any(sub.glob('*.mp4'))

    def _encode(self, path: Path, frames: np.ndarray):
        import imageio
        writer = imageio.get_writer(str(path), fps=self.fps, codec=self.codec, macro_block_size=2)
        for frame in frames:
            writer.append_data(np.repeat(frame, 3, axis=-1) if frame.shape[-1] == 1 else frame)
        writer.close()

    def _write_images(self, col_dir: Path, frames: np.ndarray, layout: str):
        if layout == 'single':
            self._encode(col_dir / f'ep_{self._ep_idx}.mp4', frames)
        else:
            for a in range(frames.shape[1]):
                self._encode(col_dir / f'ep_{self._ep_idx}_agent_{a}.mp4', frames[:, a])


@register_format
class Video(Format):
    "A folder with `ep_len.npz` and `.mp4` image columns."
    name = 'video'

    @classmethod
    def detect(cls, path) -> bool:
        p = Path(path)
        if not p.is_dir() or not (p / 'ep_len.npz').exists():
            return False
        return any(sub.is_dir() and any(sub.glob('*.mp4')) for sub in p.iterdir())

    @classmethod
    def open_reader(cls, path, **kwargs) -> VideoDataset:
        return VideoDataset(path=path, **kwargs)

    @classmethod
    def open_writer(cls, path, **kwargs) -> VideoWriter:
        return VideoWriter(path, **kwargs)

### Tests

In [ ]:
import tempfile, torch
import stable_marl as sm
from stable_marl.data import HDF5Dataset, detect_format, load_dataset

ENV = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_episode_steps=6)
with tempfile.TemporaryDirectory() as tmp:
    world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=2, add_pixels=True, render_mode='rgb_array', **ENV)
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/video', episodes=3, seed=0, format='video', progress=False)
    assert detect_format(f'{tmp}/video') is Video
    assert sorted(p.name for p in Path(f'{tmp}/video/pixels').iterdir())[:2] == ['ep_0_agent_0.mp4', 'ep_0_agent_1.mp4']
    ds = load_dataset(f'{tmp}/video', num_steps=2, frameskip=2)
    item = ds[0]
    assert item['pixels'].shape == (2, 2, 28, 28, 3)                  # (T, agents, H, W, C), 28 px kept
    assert item['render'].shape == (2, 3, 28, 28)                     # one image per step: channel-first
    assert item['action'].shape == (2, 4) and item['step_idx'].tolist() == [0, 2]
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/d.h5', episodes=3, seed=0, progress=False)
    h5 = HDF5Dataset(path=f'{tmp}/d.h5', num_steps=2, frameskip=2)
    assert (item['pixels'].float() - h5[0]['pixels'].float()).abs().mean() < 12   # lossy, but the same frames
    h5.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()